<a href="https://colab.research.google.com/github/Nahom32/Cow-Anomaly-Detection/blob/main/notebooks/Cow_Localization_and_Anomaly_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ultralytics scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 34.0 MB/s eta 0:00:00


In [ ]:
!pip install torch torchvision pytorchvideo decord

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.7/132.7 kB 5.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 5.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 17.6 MB/s eta 0:00:00
  Created wheel for pytorchvideo: filename=pytorchvideo-0.1.5-py3-none-any.whl size=188686 sha256=6dd0cf50a451e887d3d32eae98e97e37e8c2343aaf42345b108d16e14ae23457
  Stored in directory: /root/.cache/pip/wheels/b3/49/dc/aab2dce83e38b59849db13a4f4ddd220e568e24b58332fb0f9
  Created wheel for fvcore: filename=fvcore-0.1.5.post20221221-py3-none-any.whl size=61397 sha256=4e72431dd9e11ce921af664675209fed2ec509dc9f8a55c0d9eee2137beaf2a7
  Stored in directory: /root/.cache/pip/wheel

In [ ]:
!pip install opencv-python tqdm pandas

In [ ]:
!pip install kagglehub

In [ ]:
import kagglehub

path = kagglehub.dataset_download("fandaoerji/cbvd-5cow-behavior-video-dataset")
print(path)
import os

for d in os.listdir(path):
    print(d)

100%|██████████| 10.8G/10.8G [02:02<00:00, 95.3MB/s]

Extracting files...


/root/.cache/kagglehub/datasets/fandaoerji/cbvd-5cow-behavior-video-dataset/versions/11
rawframes_mini
videos_add
labelframes
labelframes_add
miniannotations
videos
CBVD-5.csv
annotations
minilabelframes


In [ ]:
print("Dataset root path:", path)


print("\nDataset structure preview:")
for root, dirs, files in os.walk(path):
    if files:
        rel = os.path.relpath(root, path)
        print(f"{rel}/  → {len(files)} files (first 3: {files[:3]})")


DATA_ROOT = path
ANNOTATIONS_DIR = os.path.join(path, "annotations")
VIDEO_DIR = os.path.join(path, "video_cut")
RAWFRAMES_DIR = os.path.join(path, "rawframes")

print(f"\n Paths ready:")
print(f"   DATA_ROOT       = {DATA_ROOT}")
print(f"   ANNOTATIONS_DIR = {ANNOTATIONS_DIR}")
print(f"   VIDEO_DIR       = {VIDEO_DIR}")

Dataset root path: /root/.cache/kagglehub/datasets/fandaoerji/cbvd-5cow-behavior-video-dataset/versions/11

Dataset structure preview:
./  → 1 files (first 3: ['CBVD-5.csv'])
rawframes_mini/139/  → 300 files (first 3: ['img_00120.jpg', 'img_00094.jpg', 'img_00160.jpg'])
rawframes_mini/368/  → 300 files (first 3: ['img_00120.jpg', 'img_00094.jpg', 'img_00160.jpg'])
rawframes_mini/216/  → 300 files (first 3: ['img_00120.jpg', 'img_00094.jpg', 'img_00160.jpg'])
rawframes_mini/453/  → 300 files (first 3: ['img_00120.jpg', 'img_00094.jpg', 'img_00160.jpg'])
rawframes_mini/671/  → 300 files (first 3: ['img_00120.jpg', 'img_00094.jpg', 'img_00160.jpg'])
rawframes_mini/234/  → 300 files (first 3: ['img_00120.jpg', 'img_00094.jpg', 'img_00160.jpg'])
rawframes_mini/125/  → 300 files (first 3: ['img_00120.jpg', 'img_00094.jpg', 'img_00160.jpg'])
rawframes_mini/610/  → 300 files (first 3: ['img_00120.jpg', 'img_00094.jpg', 'img_00160.jpg'])
rawframes_mini/531/  → 300 files (first 3: ['img_00120.jp

In [ ]:
from ultralytics import YOLO
from scipy.spatial.distance import cdist
import shutil
from pathlib import Path


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
print(f"\n Paths ready:")
print(f"   DATA_ROOT       = {DATA_ROOT}")
print(f"   ANNOTATIONS_DIR = {ANNOTATIONS_DIR}")
print(f"   VIDEO_DIR       = {VIDEO_DIR}")
DATA_ROOT = path

ANNOTATIONS_DIR = os.path.join(DATA_ROOT, "miniannotations")
FRAMES_DIR = os.path.join(DATA_ROOT, "rawframes_mini")

print(os.listdir(ANNOTATIONS_DIR))
ann_file = None

# Prioritize 'ava_train_v2.1.csv' as it seems to contain actual data
if 'ava_train_v2.1.csv' in os.listdir(ANNOTATIONS_DIR):
    ann_file = os.path.join(ANNOTATIONS_DIR, 'ava_train_v2.1.csv')
else:
    for f in os.listdir(ANNOTATIONS_DIR):
        if f.endswith(".csv"):
            ann_file = os.path.join(ANNOTATIONS_DIR, f)
            break
print("Using:", ann_file)
import pandas as pd

df = pd.read_csv(ann_file, header=None, dtype={0: str})

df.columns = [
    "video_id",
    "timestamp",
    "x1", "y1", "x2", "y2",
    "action_id",
    "target_id"
]

print(df.head())
print("Total samples:", len(df))
print("Sample frame folders:")
print(os.listdir(FRAMES_DIR)[:20])
print("Sample video_ids from annotations:")
print(df["video_id"].head(20).tolist())
valid_ids = set(os.listdir(FRAMES_DIR))

print("Number of frame folders:", len(valid_ids))
df["video_id"] = df["video_id"].apply(lambda x: str(int(x)))

df = df[df["video_id"].isin(valid_ids)]

print("Filtered dataset size:", len(df))


 Paths ready:
   DATA_ROOT       = /root/.cache/kagglehub/datasets/fandaoerji/cbvd-5cow-behavior-video-dataset/versions/11
   ANNOTATIONS_DIR = /root/.cache/kagglehub/datasets/fandaoerji/cbvd-5cow-behavior-video-dataset/versions/11/annotations
   VIDEO_DIR       = /root/.cache/kagglehub/datasets/fandaoerji/cbvd-5cow-behavior-video-dataset/versions/11/video_cut
['ava_val_excluded_timestamps_v2.1.csv', 'ava_train_excluded_timestamps_v2.1.csv', 'ava_train_v2.1.csv', 'ava_test_v2.1.csv', 'ava_dense_proposals_val.FAIR.recall_93.9.pkl', 'labelmap.txt', 'ava_val_v2.1.csv', 'ava_dense_proposals_test.FAIR.recall_93.9.pkl', 'ava_test_excluded_timestamps_v2.1.csv', 'ava_dense_proposals_train.FAIR.recall_93.9.pkl', 'ava_action_list_v2.1_for_activitynet_2018.pbtxt']
Using: /root/.cache/kagglehub/datasets/fandaoerji/cbvd-5cow-behavior-video-dataset/versions/11/miniannotations/ava_train_v2.1.csv
  video_id  timestamp        x1        y1        x2        y2  action_id  \
0      618          2  0.4570

In [ ]:
import torch
from torch.utils.data import Dataset
import os, cv2, random
import numpy as np

class CowDatasetHybrid(Dataset):
    def __init__(self, df, frames_dir, normal_classes, num_frames=8, fps=25):
        self.df = df.reset_index(drop=True)
        self.frames_dir = frames_dir
        self.num_frames = num_frames
        self.fps = fps

        self.normal_df = self.df[self.df["action_id"].isin([c+1 for c in normal_classes])]
        self.anomaly_df = self.df[~self.df["action_id"].isin([c+1 for c in normal_classes])]

    def __len__(self):
        return len(self.normal_df)

    def _load_clip(self, row):
        video_id = str(int(row.video_id))
        video_folder = os.path.join(self.frames_dir, video_id)

        if not os.path.exists(video_folder):
            return torch.zeros((self.num_frames, 3, 224, 224))

        frame_files = sorted(os.listdir(video_folder))
        if len(frame_files) == 0:
            return torch.zeros((self.num_frames, 3, 224, 224))

        center_idx = int(row.timestamp * self.fps)
        center_idx = min(max(center_idx, 0), len(frame_files)-1)

        start = max(0, center_idx - self.num_frames // 2)
        selected = frame_files[start:start+self.num_frames]

        frames = []
        for f in selected:
            img = cv2.imread(os.path.join(video_folder, f))
            if img is None:
                continue

            h, w, _ = img.shape
            x1 = int(row.x1 * w)
            y1 = int(row.y1 * h)
            x2 = int(row.x2 * w)
            y2 = int(row.y2 * h)

            x1, y1 = max(0,x1), max(0,y1)
            x2, y2 = min(w,x2), min(h,y2)

            if x2 <= x1 or y2 <= y1:
                continue

            img = img[y1:y2, x1:x2]
            img = cv2.resize(img, (224, 224))

            img = img / 255.0
            img = (img - 0.5) / 0.5

            frames.append(img)

        if len(frames) == 0:
            frames = [np.zeros((224,224,3))]*self.num_frames

        if len(frames) < self.num_frames:
            frames += [frames[-1]]*(self.num_frames-len(frames))

        frames = torch.tensor(frames).permute(0,3,1,2).float()
        return frames

    def __getitem__(self, idx):
        normal_row = self.normal_df.iloc[idx]
        normal_clip = self._load_clip(normal_row)

        # pseudo anomaly: sample from anomaly set
        pseudo_row = self.anomaly_df.sample(1).iloc[0]
        pseudo_clip = self._load_clip(pseudo_row)

        return normal_clip, pseudo_clip

In [ ]:
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import random



OUTPUT_DIR = "/content/cow_detection_dataset"   # where YOLO dataset will be saved
FPS = 25                                        # as in your dataset


os.makedirs(f"{OUTPUT_DIR}/train/images", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/train/labels", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/val/images", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/val/labels", exist_ok=True)

# Split videos into train/val (80/20)
video_ids = df["video_id"].unique()
val_videos = set(random.sample(list(video_ids), k=int(0.2 * len(video_ids))))

# Process each annotation row
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Creating YOLO dataset"):
    video_id = row["video_id"]
    # Frame index = timestamp * FPS (same as in _load_clip)
    frame_idx = int(row["timestamp"] * FPS)
    # Image file name pattern: img_00001.jpg (5 digits)
    img_filename = f"img_{frame_idx:05d}.jpg"
    img_path = Path(FRAMES_DIR) / video_id / img_filename

    if not img_path.exists():
        continue  # skip if frame not present (should not happen after filtering)

    # Determine train/val split based on video_id
    target_dir = "val" if video_id in val_videos else "train"

    # Copy image to dataset folder (use a unique name: videoId_frameIdx.jpg)
    img_dest = Path(OUTPUT_DIR) / target_dir / "images" / f"{video_id}_{frame_idx:05d}.jpg"
    img = cv2.imread(str(img_path))
    cv2.imwrite(str(img_dest), img)

    # Get image dimensions
    h, w = img.shape[:2]

    # Convert absolute bounding box to YOLO normalized format
    x1, y1, x2, y2 = row["x1"], row["y1"], row["x2"], row["y2"]
    x_center = (x1 + x2) / 2.0
    y_center = (y1 + y2) / 2.0
    box_w = x2 - x1
    box_h = y2 - y1

    # Normalize by image dimensions
    x_center_norm = x_center / w
    y_center_norm = y_center / h
    width_norm = box_w / w
    height_norm = box_h / h

    # YOLO line: class_id (0 for cow) + normalized coordinates
    yolo_line = f"0 {x_center_norm:.6f} {y_center_norm:.6f} {width_norm:.6f} {height_norm:.6f}\n"

    # Save label file with same base name as image
    label_dest = Path(OUTPUT_DIR) / target_dir / "labels" / f"{video_id}_{frame_idx:05d}.txt"
    with open(label_dest, "w") as f:
        f.write(yolo_line)

print(f"Dataset ready at {OUTPUT_DIR}")
print(f"Train images: {len(os.listdir(f'{OUTPUT_DIR}/train/images'))}")
print(f"Val images: {len(os.listdir(f'{OUTPUT_DIR}/val/images'))}")

Creating YOLO dataset: 100%|██████████| 34025/34025 [00:57<00:00, 587.05it/s]

Dataset ready at /content/cow_detection_dataset
Train images: 2335
Val images: 586


In [ ]:
from ultralytics import YOLO


# Load a pre-trained YOLO11n model
model = YOLO("yolo11n.pt")

# Train
results = model.train(
    data="/content/cow_detection.yaml",
    epochs=50,
    imgsz=640,
    batch=16,           # Reduce to 8 if you see CUDA out of memory
    cache=False,        # Important for Colab memory
    workers=2,
    device=0,           # Use GPU
    project="cow_detector",
    name="yolo11n_cbvd"
)

Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/cow_detection.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11n_cbvd-4, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=